# ANA680 Final Project
## End-to-End Machine Learning Deployment for Wine Quality Prediction

**Student:** Alberto Mendez  
**Course:** ANA680 Machine Learning Engineering

### Project Objective
Develop and evaluate a machine-learning regression model that predicts red wine quality from physicochemical measurements, save the best model, and use it in an end-to-end MLOps deployment stack involving Flask, Docker, CI/CD, Heroku, AWS SageMaker, and Kubernetes.

### Dataset
UCI Wine Quality — Red Wine dataset. The target variable is `quality` and the predictors are 11 physicochemical measurements.


In [ ]:
import json, joblib
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

DATA_PATH = Path("data/winequality-red.csv")
df = pd.read_csv(DATA_PATH, sep=";")
df.head()


## Data Exploration and Cleaning


In [ ]:
print("Shape:", df.shape)
display(df.info())
display(df.describe())
print("\nMissing values:\n", df.isna().sum())
print("\nDuplicate rows:", df.duplicated().sum())


In [ ]:
df = df.drop_duplicates().dropna().copy()
print("Cleaned shape:", df.shape)


## Exploratory Data Analysis


In [ ]:
df.hist(figsize=(14, 10), bins=20)
plt.tight_layout()
plt.show()


In [ ]:
corr = df.corr(numeric_only=True)
quality_corr = corr["quality"].sort_values(ascending=False)
quality_corr


## Feature Selection

All 11 physicochemical predictor variables are retained for the baseline final project because they are legitimate measured inputs available at prediction time. Correlation with quality is reviewed as part of EDA, while tree-based models can capture nonlinear effects that simple correlation may miss.


## Train / Validation / Test Split


In [ ]:
X = df.drop(columns=["quality"])
y = df["quality"]

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42
)

print("Training:", len(X_train))
print("Validation:", len(X_val))
print("Testing:", len(X_test))


## Model Training and Validation


In [ ]:
models = {
    "Linear Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LinearRegression())
    ]),
    "Random Forest": RandomForestRegressor(n_estimators=300, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42)
}

rows = []
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_val)
    rows.append({
        "Model": name,
        "MAE": mean_absolute_error(y_val, pred),
        "RMSE": mean_squared_error(y_val, pred) ** 0.5,
        "R2": r2_score(y_val, pred)
    })

results = pd.DataFrame(rows).sort_values("RMSE")
results


## Select and Evaluate the Best Model


In [ ]:
best_name = results.iloc[0]["Model"]
best_model = models[best_name]

X_train_final = pd.concat([X_train, X_val])
y_train_final = pd.concat([y_train, y_val])
best_model.fit(X_train_final, y_train_final)

test_pred = best_model.predict(X_test)
test_metrics = {
    "MAE": mean_absolute_error(y_test, test_pred),
    "RMSE": mean_squared_error(y_test, test_pred) ** 0.5,
    "R2": r2_score(y_test, test_pred)
}

print("Best model:", best_name)
print(test_metrics)


## Save Model and Metrics


In [ ]:
joblib.dump(best_model, "wine_quality_model.pkl")

metrics_output = {
    "best_model": best_name,
    "validation_results": results.to_dict(orient="records"),
    "test_metrics": {k: float(v) for k, v in test_metrics.items()},
    "features": list(X.columns)
}
with open("model_metrics.json", "w") as f:
    json.dump(metrics_output, f, indent=2)

print("Saved wine_quality_model.pkl")
print("Saved model_metrics.json")


## Recorded Training Results
- **Best model:** Random Forest
- **Test MAE:** 0.4600
- **Test RMSE:** 0.5950
- **Test R²:** 0.4375
- **Rows after cleaning:** 1359

These values were generated from the included UCI Red Wine Quality dataset.


## Conclusion

This notebook establishes the machine-learning foundation for the ANA680 final project. After validation-based model selection, the chosen algorithm is refit using the combined training and validation data and evaluated once on the held-out test set. The resulting serialized model is used by the Flask application and subsequent deployment stages.
